# Disambiguation parameter optimization

> **Notebook convention:** as in [`entity_linking.ipynb`](entity_linking.ipynb), keep this notebook
> limited to function calls and variable definitions (experiment parameters, calls whose results are
> displayed). Every function, class and significant chunk of logic belongs in the support script
> [`disambiguation_optimization_utils.py`](disambiguation_optimization_utils.py), taking its
> dependencies as explicit arguments rather than reading notebook globals.

Explores the parameters of the disambiguation procedure ([`modules/geo_disambiguation.py`](modules/geo_disambiguation.py)),
scored on the **training split only** (`card_id` starting with `train_`).

**Objective**, in order of priority:
1. higher precision;
2. more correctly linked addresses;
3. fewer incorrectly linked addresses (wrong links that are not an ancestor of the true location).

**Method.** The GeoDB search does not depend on the disambiguator, so the training addresses are linked once
by the production pipeline, keeping the addresses the search returned (cached under `cache/`). Every
configuration then only replays the disambiguation of those addresses, in parallel worker processes.

**Stages.** Each stage starts from the best configuration of the previous one:
1. the weights of the weighted score;
2. the parent/child distance thresholds of the hierarchical likelihood;
3. the minimum population for the population order of magnitude;
4. the order of the disambiguation factors and their significance thresholds, by a greedy search;
5. the significance thresholds, keeping the production order (from the best of stage 3);
6. the order, by moving each factor of the production order up (from the best of stage 5).

`%autoreload` is enabled below, but the evaluation workers load the support script when they start:
rerun the cell creating the `evaluator` after editing it.

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import entity_linking_notebook_utils as utils
utils.setup_logging()
import disambiguation_optimization_utils as opt
import modules.entity_linking as entity_linking

In [2]:
SEARCH_INDEX_DIR = ".geo_db_search_index"
PARSED_ADDRESSES_CACHE_PATH = Path("experiments_data/entity_linking/parsed_addresses.jsonl")
# The training addresses as the GeoDB search returned them; rebuild after changing the search or the parsing
SEARCHED_FIELDS_CACHE_PATH = Path("cache/disambiguation_optimization/searched_train_fields.pkl")
OVERWRITE_SEARCH_CACHE = False
NUM_SEARCH_WORKERS = utils.default_num_workers()
# Each evaluation worker holds every searched training address (a few hundred MB)
NUM_EVALUATION_WORKERS = 32
# Addresses per evaluation task: small enough for a single configuration to keep every worker busy
EVALUATION_CHUNK_SIZE = 25

In [3]:
ground_truth = utils.load_ground_truth("open_data/bzkopen_linking_groundtruth.jsonl")
parsed_addresses = await utils.load_or_build_parsed_addresses(ground_truth, PARSED_ADDRESSES_CACHE_PATH)
train_rows, train_ground_truth = opt.train_split(parsed_addresses, ground_truth)
print(f"Training addresses: {len(train_rows)} of {len(ground_truth)}")

Running regex parsing on 1090 addresses...
606/1090 addresses need the LLM fallback (regex left them un/partially parsed).
Retrieving cached parsed addresses from experiments_data/entity_linking/parsed_addresses.jsonl...
Training addresses: 771 of 1090


In [4]:
searched_fields = opt.load_or_search_fields(
    train_rows, SEARCHED_FIELDS_CACHE_PATH, f"{SEARCH_INDEX_DIR}/tantivy_index", NUM_SEARCH_WORKERS,
    overwrite=OVERWRITE_SEARCH_CACHE,
)

Retrieving cached searched addresses from cache/disambiguation_optimization/searched_train_fields.pkl...


## Production configuration

The configuration `entity_linking.build_disambiguator()` builds. Replaying its disambiguation must link every
training address as the pipeline did when searching.

In [5]:
opt.cleanup_previous_run(globals()) # Allow script rerunning
production_config = opt.DisambiguationConfig.of(entity_linking.build_disambiguator())
evaluator = opt.ConfigEvaluator(
    SEARCHED_FIELDS_CACHE_PATH, train_ground_truth, NUM_EVALUATION_WORKERS, EVALUATION_CHUNK_SIZE)
opt.check_replay(evaluator, production_config, searched_fields)
evaluator.evaluate({"production": production_config})

2026-10-07 15:04:19.066 INFO     entity_linking.Disambiguator geo_disambiguation.py:370 Initialized with significance thresholds defaultdict(<function _default_significance_threshold at 0x7fc810fe1d00>, {'fuzzy_similarity_score': 0, 'population_order_of_magnitude': 1, 'weighted_score': 0.05, 'child_parent_likelihood': 0.05, 'entity_types_matching_preferred': 0.05, 'country_likelihood_rank': 0.05, 'phonetic_score': 0.05, 'entity_types_matching': 0.05, 'is_preferred_name': 0.05, 'population_count': 0.05}), primary priority ['weighted_score', 'child_parent_likelihood', 'population_order_of_magnitude', 'entity_types_matching_preferred', 'fuzzy_similarity_score', 'country_likelihood_rank', 'phonetic_score', 'entity_types_matching', 'is_preferred_name', 'population_count'], secondary priority [], population rounding factor 1, min population order of magnitude 100000, score prune thresholds {'fuzzy_similarity_score': 0.4, 'child_parent_likelihood': 0.4}, geo db geo.duckdb, geometry parent/chi

Evaluating 1 configurations:   0%|          | 0/31 [00:00<?, ?it/s]

Replaying the disambiguation links all 771 addresses as the pipeline did


,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
production,0.87984,659,79,11,5,0.932767,0.99247


In [6]:
print(opt.describe_config(production_config))

# modules/geo_disambiguation.py
DISAMBIGUATION_FACTOR_PRIMARY_PRIORITY = [
    'weighted_score',
    'child_parent_likelihood',
    'population_order_of_magnitude',
    'entity_types_matching_preferred',
    'fuzzy_similarity_score',
    'country_likelihood_rank',
    'phonetic_score',
    'entity_types_matching',
    'is_preferred_name',
    'population_count',
]
DISAMBIGUATION_FACTOR_SECONDARY_PRIORITY = [
]
WEIGHTED_DISAMBIGUATION_FACTORS = {
    'phonetic_score': 2,
    'fuzzy_similarity_score': 3,
    'entity_types_matching_preferred': 2,
}
PARENT_CHILD_MAX_DISTANCE_KM_BY_ADMIN_LEVEL = {
    0: 100,
    1: 100,
    2: 34,
    3: 14,
    4: 6,
    5: 5,
}
# Disambiguator(min_population_order_of_magnitude=...)
min_population_order_of_magnitude = 100_000
# modules/entity_linking.py
SIGNIFICANCE_THRESHOLDS = {
    'weighted_score': 0.05,
    'child_parent_likelihood': 0.05,
    'population_order_of_magnitude': 1,
    'entity_types_matching_preferred': 0.05,
    'fuzzy_similarity_score

In [7]:
base_congig = production_config.replace(
    primary_priority = production_config.primary_priority + production_config.secondary_priority,
    secondary_priority = tuple()
)
evaluator.evaluate({"base": production_config})

,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
base,0.87984,659,79,11,5,0.932767,0.99247


## Stage 1: weights of the weighted score

The same four factors as the production weights. Round 1 sets every weight to 1 and tries a weight of 2 for each
factor in turn. Round 2 gives the factor that did best at 2 a weight of 3, and tries the other factors at 2 except
one, which weighs 1 or is excluded in turn; plus the other factors all at 2, and all at 1. The production weights
and all weights at 1 are evaluated as well. The best configuration of round 2 is the best overall, since round 2
includes the best of round 1.

In [8]:
weights_round_1, weights_round_2 = opt.optimize_score_weights(evaluator, base_congig)
weights_round_1.evaluations

Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

Best factor weighing 2 in round 1: fuzzy_similarity_score


,phonetic_score,fuzzy_similarity_score,entity_types_matching_preferred,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
"phonetic_score=1, fuzzy_similarity_score=2, entity_types_matching_preferred=1",1,2,1,0.879840,659,78,12,5,0.932767,0.992470
current,2,3,2,0.879840,659,79,11,5,0.932767,0.992470
"phonetic_score=1, fuzzy_similarity_score=1, entity_types_matching_preferred=1",1,1,1,0.878505,658,80,11,5,0.932011,0.992459
"phonetic_score=2, fuzzy_similarity_score=1, entity_types_matching_preferred=1",2,1,1,0.874499,655,83,11,5,0.929737,0.992424
"phonetic_score=1, fuzzy_similarity_score=1, entity_types_matching_preferred=2",1,1,2,0.873164,654,84,11,5,0.928977,0.992413


In [9]:
weights_round_2.evaluations

,phonetic_score,fuzzy_similarity_score,entity_types_matching_preferred,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
"phonetic_score=1, fuzzy_similarity_score=2, entity_types_matching_preferred=1",1,2,1,0.879840,659,78,12,5,0.932767,0.992470
"phonetic_score=1, fuzzy_similarity_score=3, entity_types_matching_preferred=1",1,3,1,0.879840,659,78,12,5,0.932767,0.992470
"phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=1",2,3,1,0.879840,659,78,12,5,0.932767,0.992470
"phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=2",2,3,2,0.879840,659,79,11,5,0.932767,0.992470
"phonetic_score=1, fuzzy_similarity_score=1, entity_types_matching_preferred=1",1,1,1,0.878505,658,80,11,5,0.932011,0.992459
"phonetic_score=1, fuzzy_similarity_score=3, entity_types_matching_preferred=2",1,3,2,0.877170,657,80,12,5,0.931254,0.992447
"phonetic_score=0, fuzzy_similarity_score=3, entity_types_matching_preferred=2",0,3,2,0.874499,655,82,12,5,0.929737,0.992424
"phonetic_score=2, fuzzy_similarity_score=3, entity_types_matching_preferred=0",2,3,0,0.859813,644,85,20,5,0.921316,0.992296


In [10]:
after_weights = weights_round_2.best_config
print(f"Best weights: {after_weights.score_weights}")

Best weights: {'phonetic_score': 1, 'fuzzy_similarity_score': 2, 'entity_types_matching_preferred': 1}


## Stage 2: distance thresholds of the hierarchical likelihood

Distance from a parent beyond which a child is considered outside it, by the parent's admin level
(`PARENT_CHILD_MAX_DISTANCE_KM_BY_ADMIN_LEVEL`). The production cascade is scaled so that the first threshold
(a region or a state) is each of the distances below, keeping the ratios between levels.

In [11]:
FIRST_DISTANCES_KM = [100, 200, 300, 400, 500]
distances = opt.optimize_parent_child_distances(evaluator, after_weights, FIRST_DISTANCES_KM)
distances.evaluations

Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

,admin level 0 (km),admin level 1 (km),admin level 2 (km),admin level 3 (km),admin level 4 (km),admin level 5 (km),precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
100 / 100 / 34 / 14 / 6 / 5,100,100,34,14,6,5,0.879840,659,78,12,5,0.932767,0.992470
200 / 200 / 68 / 28 / 12 / 10,200,200,68,28,12,10,0.878505,658,79,12,5,0.932011,0.992459
300 / 300 / 102 / 42 / 18 / 15,300,300,102,42,18,15,0.874499,655,81,13,5,0.929737,0.992424
400 / 400 / 136 / 56 / 24 / 20,400,400,136,56,24,20,0.874499,655,82,12,5,0.929737,0.992424
500 / 500 / 170 / 70 / 30 / 25,500,500,170,70,30,25,0.873164,654,83,12,5,0.928977,0.992413


In [12]:
after_distances = distances.best_config
print(f"Best distances: {after_distances.parent_child_max_distance_km_by_admin_level}")

Best distances: {0: 100, 1: 100, 2: 34, 3: 14, 4: 6, 5: 5}


## Stage 3: minimum population for the population order of magnitude

Population under which `population_order_of_magnitude` is 0.

In [13]:
MIN_POPULATIONS = [0, 100, 1_000, 10_000, 100_000, 500_000, 1_000_000]
population = opt.optimize_min_population_order_of_magnitude(evaluator, after_distances, MIN_POPULATIONS)
population.evaluations

Evaluating 6 configurations:   0%|          | 0/186 [00:00<?, ?it/s]

,min_population_order_of_magnitude,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
100_000,100000,0.879840,659,78,12,5,0.932767,0.992470
500_000,500000,0.875834,656,78,15,5,0.930496,0.992436
1_000_000,1000000,0.874499,655,79,15,5,0.929737,0.992424
1_000,1000,0.873164,654,87,8,5,0.928977,0.992413
10_000,10000,0.871829,653,86,10,5,0.928216,0.992401
100,100,0.867824,650,91,8,5,0.925926,0.992366
0,0,0.866489,649,92,8,5,0.925160,0.992355


In [14]:
after_population = population.best_config
print(f"Best min population: {after_population.min_population_order_of_magnitude:_}")

Best min population: 100_000


## Stage 4: order of the factors, by a greedy search

Which factor alone decides best? With that one set, which factor decides best after it, and with which
significance threshold on the first? This repeats until all 10 factors are ordered, and a final step picks
the threshold of the last factor. The factor just added gets a threshold of 0 until the next step sets it.

The ordered factors are all primary factors: candidates no factor tells apart by its threshold are compared
again on the same factors with no threshold (see `geo_disambiguation.comparison_steps`); there are no
secondary factors.

Each factor gets 5 threshold candidates suited to the range of its address-level score (an average over the
address' entities):
- scores between 0 and 1 (`weighted_score`, `fuzzy_similarity_score`, `phonetic_score`): fine steps around the production 0.05;
- `child_parent_likelihood` and the entity type matches: coarser steps, since per-entity values are few
  (e.g. 0, 0.25, 0.75, 1) or are ratios of admin codes;
- `is_preferred_name`: 0 or 1 per entity;
- `country_likelihood_rank`: 0 to 3 per entity;
- `population_order_of_magnitude`: 0 or the rounded log10 of the population (about 2 to 7);
- `population_count`: the raw population (`DISAMBIGUATION_POPULATION_ROUNDING_FACTOR` is 1).

In [15]:
SIGNIFICANCE_THRESHOLD_CANDIDATES = {
    # in the production priority order, which breaks ties between orders
    "weighted_score": [0, 0.025, 0.05, 0.1, 0.2],
    "child_parent_likelihood": [0, 0.05, 0.1, 0.2, 0.3],
    "fuzzy_similarity_score": [0, 0.025, 0.05, 0.1, 0.2],
    "entity_types_matching_preferred": [0, 0.05, 0.1, 0.2, 0.3],
    "population_order_of_magnitude": [0, 0.25, 0.5, 1, 2],
    "country_likelihood_rank": [0, 0.25, 0.5, 1, 1.5],
    "phonetic_score": [0, 0.025, 0.05, 0.1, 0.2],
    "entity_types_matching": [0, 0.05, 0.1, 0.2, 0.3],
    "is_preferred_name": [0, 0.1, 0.25, 0.5, 0.75],
    "population_count": [0, 1_000, 10_000, 100_000, 1_000_000],
}

In [16]:
order_search = opt.greedy_order_search(evaluator, after_population, SIGNIFICANCE_THRESHOLD_CANDIDATES)
order_search.summary

Evaluating 10 configurations:   0%|          | 0/310 [00:00<?, ?it/s]

Step: weighted_score (0)


Evaluating 45 configurations:   0%|          | 0/1395 [00:00<?, ?it/s]

Step: weighted_score (0.025) > population_count (0)


Evaluating 40 configurations:   0%|          | 0/1240 [00:00<?, ?it/s]

...

Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

Step: weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0) > country_likelihood_rank (0)


Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

Step: weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0) > country_likelihood_rank (0)
Best order (step 9): weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0)


,added factor,threshold set on,threshold,configurations evaluated,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
step,,,,,,,,,,,
1,weighted_score,NaN,NaN,10,0.608621,353,63,164,174,0.637760,0.669829
2,population_count,weighted_score,0.025,45,0.811245,606,130,11,7,0.891176,0.988581
3,entity_types_matching,population_count,1000.000,40,0.832664,622,112,13,7,0.904070,0.988871
4,fuzzy_similarity_score,entity_types_matching,0.000,35,0.842246,630,106,12,6,0.910405,0.990566
5,is_preferred_name,fuzzy_similarity_score,0.000,30,0.847390,633,103,11,7,0.912761,0.989062
6,child_parent_likelihood,is_preferred_name,0.000,25,0.850067,635,105,7,7,0.914327,0.989097
7,entity_types_matching_preferred,child_parent_likelihood,0.000,20,0.850067,635,105,7,7,0.914327,0.989097
8,population_order_of_magnitude,entity_types_matching_preferred,0.000,15,0.850067,635,105,7,7,0.914327,0.989097
9,phonetic_score,population_order_of_magnitude,0.000,10,0.850067,635,105,7,7,0.914327,0.989097


In [17]:
opt.display_greedy_steps(order_search, top=5)

**Step 1**: factor added after nan, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score (0),weighted_score,None,0.608621,353,63,164,174,0.637760,0.669829
population_count (0),population_count,None,0.579439,434,244,71,5,0.730640,0.988610
fuzzy_similarity_score (0),fuzzy_similarity_score,None,0.552817,314,48,206,186,0.588015,0.628000
population_order_of_magnitude (0),population_order_of_magnitude,None,0.541012,310,83,180,181,0.582707,0.631365
phonetic_score (0),phonetic_score,None,0.533835,284,49,199,222,0.547206,0.561265


**Step 2**: factor added after weighted_score, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score (0.025) > population_count (0),population_count,0.025,0.811245,606,130,11,7,0.891176,0.988581
weighted_score (0) > population_count (0),population_count,0.000,0.809651,604,129,13,8,0.889543,0.986928
weighted_score (0.05) > population_count (0),population_count,0.050,0.803476,601,140,7,6,0.887085,0.990115
weighted_score (0.1) > population_count (0),population_count,0.100,0.770360,577,163,9,5,0.867017,0.991409
weighted_score (0.05) > population_order_of_magnitude (0),population_order_of_magnitude,0.050,0.751067,528,71,104,51,0.823713,0.911917


**Step 3**: factor added after population_count, and the threshold of the latter

...

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0),phonetic_score,0.00,0.850067,635,105,7,7,0.914327,0.989097
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0.25) > phonetic_score (0),phonetic_score,0.25,0.850067,635,105,7,7,0.914327,0.989097
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0.5) > phonetic_score (0),phonetic_score,0.50,0.850067,635,105,7,7,0.914327,0.989097
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (1) > phonetic_score (0),phonetic_score,1.00,0.850067,635,105,7,7,0.914327,0.989097
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (2) > phonetic_score (0),phonetic_score,2.00,0.850067,635,105,7,7,0.914327,0.989097


**Step 10**: factor added after phonetic_score, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0) > country_likelihood_rank (0),country_likelihood_rank,0.000,0.84893,635,105,8,6,0.914327,0.99064
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0.025) > country_likelihood_rank (0),country_likelihood_rank,0.025,0.84893,635,105,8,6,0.914327,0.99064
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0.05) > country_likelihood_rank (0),country_likelihood_rank,0.050,0.84893,635,105,8,6,0.914327,0.99064
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0.1) > country_likelihood_rank (0),country_likelihood_rank,0.100,0.84893,635,105,8,6,0.914327,0.99064
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0.2) > country_likelihood_rank (0),country_likelihood_rank,0.200,0.84893,635,105,8,6,0.914327,0.99064


**Step 11**: factor added after country_likelihood_rank, and the threshold of the latter

,added factor,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0) > country_likelihood_rank (0),None,0.00,0.84893,635,105,8,6,0.914327,0.99064
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0) > country_likelihood_rank (0.25),None,0.25,0.84893,635,105,8,6,0.914327,0.99064
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0) > country_likelihood_rank (0.5),None,0.50,0.84893,635,105,8,6,0.914327,0.99064
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0) > country_likelihood_rank (1),None,1.00,0.84893,635,105,8,6,0.914327,0.99064
weighted_score (0.025) > population_count (1000) > entity_types_matching (0) > fuzzy_similarity_score (0) > is_preferred_name (0) > child_parent_likelihood (0) > entity_types_matching_preferred (0) > population_order_of_magnitude (0) > phonetic_score (0) > country_likelihood_rank (1.5),None,1.50,0.84893,635,105,8,6,0.914327,0.99064


## Stage 5: significance thresholds, in the production order

Keeps the production order of the factors (primary, then secondary) and the best configuration of stage 3. Greedily,
from the first factor to the last, each factor tries the 5 threshold candidates of stage 4, with the thresholds
already chosen for the factors before it. The factor's current threshold (0.05 in production) is tried too, and
kept among ties.

In [18]:
threshold_search = opt.optimize_significance_thresholds(evaluator, after_population, SIGNIFICANCE_THRESHOLD_CANDIDATES)
threshold_search.summary

Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

weighted_score: 0.05 -> 0.05


Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

child_parent_likelihood: 0.05 -> 0.05


Evaluating 4 configurations:   0%|          | 0/124 [00:00<?, ?it/s]

...

Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

is_preferred_name: 0.05 -> 0.05


Evaluating 5 configurations:   0%|          | 0/155 [00:00<?, ?it/s]

population_count: 0.05 -> 0.05


,previous threshold,threshold,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
factor,,,,,,,,,
weighted_score,0.05,0.05,0.87984,659,78,12,5,0.932767,0.99247
child_parent_likelihood,0.05,0.05,0.87984,659,78,12,5,0.932767,0.99247
population_order_of_magnitude,1.00,1.00,0.87984,659,78,12,5,0.932767,0.99247
entity_types_matching_preferred,0.05,0.05,0.87984,659,78,12,5,0.932767,0.99247
fuzzy_similarity_score,0.00,0.00,0.87984,659,78,12,5,0.932767,0.99247
country_likelihood_rank,0.05,0.05,0.87984,659,78,12,5,0.932767,0.99247
phonetic_score,0.05,0.05,0.87984,659,78,12,5,0.932767,0.99247
entity_types_matching,0.05,0.05,0.87984,659,78,12,5,0.932767,0.99247
is_preferred_name,0.05,0.05,0.87984,659,78,12,5,0.932767,0.99247


In [19]:
opt.display_sequential_steps(threshold_search)

**weighted_score**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
weighted_score = 0.05,0.050,True,0.879840,659,78,12,5,0.932767,0.992470
weighted_score = 0.1,0.100,False,0.874499,655,83,11,5,0.929737,0.992424
weighted_score = 0.025,0.025,False,0.873164,654,83,12,5,0.928977,0.992413
weighted_score = 0,0.000,False,0.871658,652,82,14,6,0.927454,0.990881
weighted_score = 0.2,0.200,False,0.850467,637,94,18,5,0.915888,0.992212


**child_parent_likelihood**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
child_parent_likelihood = 0.05,0.05,True,0.879840,659,78,12,5,0.932767,0.992470
child_parent_likelihood = 0.1,0.10,False,0.879840,659,78,12,5,0.932767,0.992470
child_parent_likelihood = 0.2,0.20,False,0.873164,654,82,13,5,0.928977,0.992413
child_parent_likelihood = 0,0.00,False,0.873164,654,83,12,5,0.928977,0.992413
child_parent_likelihood = 0.3,0.30,False,0.870494,652,84,13,5,0.927454,0.992390


**population_order_of_magnitude**

...

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
entity_types_matching = 0.05,0.05,True,0.87984,659,78,12,5,0.932767,0.99247
entity_types_matching = 0,0.00,False,0.87984,659,78,12,5,0.932767,0.99247
entity_types_matching = 0.1,0.10,False,0.87984,659,78,12,5,0.932767,0.99247
entity_types_matching = 0.2,0.20,False,0.87984,659,78,12,5,0.932767,0.99247
entity_types_matching = 0.3,0.30,False,0.87984,659,78,12,5,0.932767,0.99247


**is_preferred_name**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
is_preferred_name = 0.05,0.05,True,0.87984,659,78,12,5,0.932767,0.992470
is_preferred_name = 0,0.00,False,0.87984,659,78,12,5,0.932767,0.992470
is_preferred_name = 0.1,0.10,False,0.87984,659,78,12,5,0.932767,0.992470
is_preferred_name = 0.25,0.25,False,0.87984,659,78,12,5,0.932767,0.992470
is_preferred_name = 0.5,0.50,False,0.87717,657,80,12,5,0.931254,0.992447
is_preferred_name = 0.75,0.75,False,0.87717,657,80,12,5,0.931254,0.992447


**population_count**

,threshold,current,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
population_count = 0.05,0.05,True,0.879840,659,78,12,5,0.932767,0.992470
population_count = 0,0.00,False,0.879840,659,78,12,5,0.932767,0.992470
population_count = 1000,1000.00,False,0.878505,658,78,13,5,0.932011,0.992459
population_count = 10000,10000.00,False,0.878505,658,78,13,5,0.932011,0.992459
population_count = 100000,100000.00,False,0.878505,658,78,13,5,0.932011,0.992459
population_count = 1e+06,1000000.00,False,0.878505,658,78,13,5,0.932011,0.992459


In [20]:
after_thresholds = threshold_search.best_config

## Stage 6: order, by moving factors up

Starts from the best configuration of stage 5. From the last factor of the production order to the first, each
factor moves up one position at a time while the objective is not worse than at the position before (ties keep
moving it up). It stays at the last position that was not worse. The boundary between primary and secondary
factors stays at the same position: a secondary factor moved above it becomes primary, and the last primary factor
becomes secondary.

In [21]:
order_mutation = opt.move_factors_up(evaluator, after_thresholds)
order_mutation.summary

Evaluating 9 configurations:   0%|          | 0/279 [00:00<?, ?it/s]

population_count: position 10 -> 10


Evaluating 8 configurations:   0%|          | 0/248 [00:00<?, ?it/s]

is_preferred_name: position 9 -> 9


Evaluating 7 configurations:   0%|          | 0/217 [00:00<?, ?it/s]

...

Evaluating 2 configurations:   0%|          | 0/62 [00:00<?, ?it/s]

population_order_of_magnitude: position 3 -> 3


Evaluating 1 configurations:   0%|          | 0/31 [00:00<?, ?it/s]

child_parent_likelihood: position 2 -> 2
weighted_score: position 1 -> 1


,from position,to position,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
factor,,,,,,,,,
population_count,10,10,0.87984,659.0,78.0,12.0,5.0,0.932767,0.99247
is_preferred_name,9,9,0.87984,659.0,78.0,12.0,5.0,0.932767,0.99247
entity_types_matching,8,8,0.87984,659.0,78.0,12.0,5.0,0.932767,0.99247
phonetic_score,7,7,0.87984,659.0,78.0,12.0,5.0,0.932767,0.99247
country_likelihood_rank,6,6,0.87984,659.0,78.0,12.0,5.0,0.932767,0.99247
fuzzy_similarity_score,5,5,0.87984,659.0,78.0,12.0,5.0,0.932767,0.99247
entity_types_matching_preferred,4,4,0.87984,659.0,78.0,12.0,5.0,0.932767,0.99247
population_order_of_magnitude,3,3,0.87984,659.0,78.0,12.0,5.0,0.932767,0.99247
child_parent_likelihood,2,2,0.87984,659.0,78.0,12.0,5.0,0.932767,0.99247


In [22]:
opt.display_sequential_steps(order_mutation)

**population_count**

,position,worse or same as the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
population_count at position 9,9,True,0.871829,653,85,11,5,0.928216,0.992401
population_count at position 8,8,False,0.850467,637,104,8,5,0.915888,0.992212
population_count at position 7,7,False,0.851802,638,104,7,5,0.916667,0.992224
population_count at position 6,6,False,0.841121,630,112,7,5,0.910405,0.992126
population_count at position 5,5,False,0.825100,618,124,7,5,0.900875,0.991974
population_count at position 4,4,False,0.825100,618,124,7,5,0.900875,0.991974
population_count at position 3,3,False,0.825100,618,124,7,5,0.900875,0.991974
population_count at position 2,2,False,0.813084,609,133,7,5,0.893617,0.991857
population_count at position 1,1,False,0.594126,445,256,48,5,0.742285,0.988889


**is_preferred_name**

,position,worse or same as the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
is_preferred_name at position 8,8,True,0.879840,659,78,12,5,0.932767,0.992470
is_preferred_name at position 7,7,False,0.879679,658,79,11,6,0.932011,0.990964
is_preferred_name at position 6,6,False,0.877005,656,81,11,6,0.930496,0.990937
is_preferred_name at position 5,5,False,0.855422,639,97,11,7,0.917444,0.989164
is_preferred_name at position 4,4,False,0.855422,639,97,11,7,0.917444,0.989164
is_preferred_name at position 3,3,False,0.849866,634,100,12,8,0.913545,0.987539
is_preferred_name at position 2,2,False,0.843164,629,104,13,8,0.909617,0.987441
is_preferred_name at position 1,1,False,0.809140,602,126,16,10,0.887906,0.983660


**entity_types_matching**

...

,position,worse or same as the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
population_order_of_magnitude at position 2,2,True,0.878505,658,79,12,5,0.932011,0.992459
population_order_of_magnitude at position 1,1,False,0.825100,618,112,19,5,0.900875,0.991974


**child_parent_likelihood**

,position,worse or same as the position below,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall
child_parent_likelihood at position 1,1,True,0.867824,650,87,12,5,0.925926,0.992366


**weighted_score**

_already first_

In [23]:
after_order_mutation = order_mutation.best_config

## Summary

In [24]:
stage_configs = {
    "production": production_config,
    "base" : base_congig,
    "1. weights": after_weights,
    "2. distances": after_distances,
    "3. min population": after_population,
    "4. greedy factor order": order_search.best_config,
    "5. thresholds": after_thresholds,
    "6. moving factors up": after_order_mutation,
}
opt.compare_stages(evaluator, stage_configs)

,precision,correctly_linked,incorrectly_linked,partially_linked,failed_to_link,f1,recall,Δ precision,Δ correctly_linked,Δ incorrectly_linked
production,0.879840,659,79,11,5,0.932767,0.992470,0.000000,0.0,0.0
base,0.879840,659,79,11,5,0.932767,0.992470,0.000000,0.0,0.0
1. weights,0.879840,659,78,12,5,0.932767,0.992470,0.000000,0.0,-1.0
2. distances,0.879840,659,78,12,5,0.932767,0.992470,0.000000,0.0,-1.0
3. min population,0.879840,659,78,12,5,0.932767,0.992470,0.000000,0.0,-1.0
4. greedy factor order,0.850067,635,105,7,7,0.914327,0.989097,-0.029773,-24.0,26.0
5. thresholds,0.879840,659,78,12,5,0.932767,0.992470,0.000000,0.0,-1.0
6. moving factors up,0.879840,659,78,12,5,0.932767,0.992470,0.000000,0.0,-1.0


In [25]:
opt.deciding_factor_counts(evaluator, stage_configs)

,production,base,1. weights,2. distances,3. min population,4. greedy factor order,5. thresholds,6. moving factors up
(common parent of tied candidates),9,9,10,10,10,6,10,10
(single candidate),93,93,93,93,93,93,93,93
child_parent_likelihood,40,40,39,39,39,6,39,39
country_likelihood_rank,25,25,25,25,25,0,25,25
entity_types_matching,27,27,27,27,27,20,27,27
fuzzy_similarity_score,37,37,32,32,32,6,32,32
is_preferred_name,7,7,7,7,7,5,7,7
phonetic_score,9,9,9,9,9,4,9,9
population_count,60,60,61,61,61,292,61,61
population_count (no threshold),0,0,0,0,0,14,0,0


In [26]:
print(opt.describe_config(after_order_mutation))

# modules/geo_disambiguation.py
DISAMBIGUATION_FACTOR_PRIMARY_PRIORITY = [
    'weighted_score',
    'child_parent_likelihood',
    'population_order_of_magnitude',
    'entity_types_matching_preferred',
    'fuzzy_similarity_score',
    'country_likelihood_rank',
    'phonetic_score',
    'entity_types_matching',
    'is_preferred_name',
    'population_count',
]
DISAMBIGUATION_FACTOR_SECONDARY_PRIORITY = [
]
WEIGHTED_DISAMBIGUATION_FACTORS = {
    'phonetic_score': 1,
    'fuzzy_similarity_score': 2,
    'entity_types_matching_preferred': 1,
}
PARENT_CHILD_MAX_DISTANCE_KM_BY_ADMIN_LEVEL = {
    0: 100,
    1: 100,
    2: 34,
    3: 14,
    4: 6,
    5: 5,
}
# Disambiguator(min_population_order_of_magnitude=...)
min_population_order_of_magnitude = 100_000
# modules/entity_linking.py
SIGNIFICANCE_THRESHOLDS = {
    'weighted_score': 0.05,
    'child_parent_likelihood': 0.05,
    'population_order_of_magnitude': 1,
    'entity_types_matching_preferred': 0.05,
    'fuzzy_similarity_score

In [27]:
evaluator.close()